# 06 — Generator Selection

Manual, validation-only selection of one eligible fine-tuned and one eligible from-scratch generator. There is no approval script, signature, Git gate, or hidden automatic winner.

## Load benchmark results and registry

In [ ]:
from pathlib import Path
import json
import sys
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))
import csv
from notebooks.utility.generator_benchmark import load_protocol, load_registry, save_selected_generators, validate_selected_generators
protocol = load_protocol(ROOT)
registry = load_registry(ROOT)
metrics_path = ROOT / protocol['outputs']['metrics']
benchmark_rows = list(csv.DictReader(metrics_path.open())) if metrics_path.is_file() else []
benchmark_rows[:3] if benchmark_rows else 'Not yet evaluated'

## Eligible family candidates and technical gates

In [ ]:
by_id = {entry['id']: entry for entry in registry['generators']}
eligible_finetuned = [entry for entry in registry['generators'] if entry['scientific_family'] == 'finetuned' and entry['eligible_for_downstream_selection']]
eligible_from_scratch = [entry for entry in registry['generators'] if entry['scientific_family'] == 'from_scratch' and entry['eligible_for_downstream_selection']]
{'finetuned': [entry['id'] for entry in eligible_finetuned], 'from_scratch': [entry['id'] for entry in eligible_from_scratch]}

## KID intervals, Coverage, Precision, FID, train memorization, duplicates and efficiency

In [ ]:
display_columns = ['generator_id', 'candidate_role', 'raddino_kid_mean', 'raddino_kid_2_5', 'raddino_kid_97_5',
                   'coverage', 'precision', 'fid_descriptive', 'train_memorization_rate',
                   'synthetic_duplicate_rate', 'generation_time_per_image']
display_columns

## Practical equivalence and manual decision

In [ ]:
SELECTED_FINETUNED_GENERATOR = None
SELECTED_FROM_SCRATCH_GENERATOR = None
SELECTION_NOTES = 'Explain practical equivalence, gates, KID intervals, diversity and efficiency here.'

## Validate and save the simple selection file

In [ ]:
SAVE_SELECTION = False
if SAVE_SELECTION:
    selected = validate_selected_generators(SELECTED_FINETUNED_GENERATOR, SELECTED_FROM_SCRATCH_GENERATOR,
                                            registry, benchmark_rows, protocol['synthetic_pool_target'])
    output = save_selected_generators(ROOT, selected['finetuned'], selected['from_scratch'], benchmark_rows,
                                      notes=SELECTION_NOTES)
    print(output)
else:
    print('Selection not saved. Set both IDs after reviewing real benchmark results, then set SAVE_SELECTION=True.')